In [68]:
import pandas as pd
import numpy as np
import sys
import os
from dotenv import load_dotenv
import json
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.inspection import permutation_importance

In [2]:
pd.set_option("display.max_columns", 100)
load_dotenv()
data_path = os.getenv("DATA_PATH")
src_path = os.getenv("SRC_PATH")
transaction_path = os.path.join(data_path, r"raw/train_transaction.csv/train_transaction.csv")
identity_path = os.path.join(data_path, r"raw/train_identity.csv/train_identity.csv")
transaction_df = pd.read_csv(transaction_path)
identity_df = pd.read_csv(identity_path)
full_df = transaction_df.merge(identity_df, on="TransactionID", how="left")
full_df = full_df.sort_values("TransactionDT").reset_index(drop=True)

In [42]:
sys.path.append(src_path)
from features.engineering import create_d_features
from data.split import temporal_split
from model.preprocessor_pipe_evalueate import evaluate_model, create_pipeline

In [53]:
json_path = os.path.join(data_path, r"processed/split_info.json")
with open(json_path, "r") as f:
    split_info = json.load(f)
    
train_end = split_info.get("train_end")
val_end = split_info.get("validation_end")

train, val, test = temporal_split(full_df, train_end, val_end)

In [52]:
test

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,addr1,addr2,dist1,dist2,P_emaildomain,R_emaildomain,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10,C11,C12,C13,C14,D1,D2,D3,D4,D5,D6,D7,D8,D9,D10,D11,D12,D13,D14,D15,M1,M2,M3,M4,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339,id_01,id_02,id_03,id_04,id_05,id_06,id_07,id_08,id_09,id_10,id_11,id_12,id_13,id_14,id_15,id_16,id_17,id_18,id_19,id_20,id_21,id_22,id_23,id_24,id_25,id_26,id_27,id_28,id_29,id_30,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo


In [54]:
map_dfs = {"train": train, "val": val, "test": test}

In [55]:
d_features_dfs = {}
for name, sample_df in map_dfs.items():
    d_features_dfs[name] = create_d_features(sample_df)

In [9]:
print("Total features:", len(d_features_dfs["train"].columns))
print(d_features_dfs["train"].columns.tolist())

Total features: 37
['TransactionDT', 'TransactionAmt', 'ProductCD', 'P_emaildomain', 'R_emaildomain', 'card1', 'card2', 'card4', 'card6', 'addr1', 'addr2', 'dist1', 'dist2', 'missing_count', 'missing_V_count', 'missing_D_count', 'missing_M_count', 'missing_C_count', 'missing_id_count', 'missing_card_count', 'missing_addr_count', 'missing_dist_count', 'transaction_day', 'transaction_hour', 'transaction_amt_log', 'amount_decimal', 'hour_sin', 'hour_cos', 'P_emaildomain_is_missing', 'P_emaildomain_provider', 'R_emaildomain_is_missing', 'R_emaildomain_provider', 'domain_math', 'is_unusual_email', 'card_product', 'email_product', 'card_addr']


In [27]:
categorical_cols = d_features_dfs["train"].select_dtypes(include='object').columns.tolist()

In [37]:
def categoricl_fraud_rate(df, column, min_count = 100):
    products_stats = (df.groupby(column)['isFraud'].agg(["mean", "count"]).rename(columns = {"mean": "fraud_rate"}))
    products_stats['fraud_rate'] = products_stats['fraud_rate'] * 100

    return products_stats[products_stats['count'] >= min_count].sort_values("fraud_rate", ascending=False)

In [56]:
X_train = d_features_dfs["train"]
X_val = d_features_dfs["val"]

y_train = train["isFraud"]
y_val = val["isFraud"]

pipe = create_pipeline(X_train)

pipe.fit(X_train, y_train)

metrics = evaluate_model(pipe, X_val, y_val)

In [61]:
metrics

{'pr_auc': 0.23076717327719434,
 'roc_auc': 0.8317765385769453,
 'precision': 0.11486058301647656,
 'recall': 0.7149901380670611,
 'f1': 0.19792519792519792}

In [60]:
preprocessor = pipe.named_steps['preprocessor']
model = pipe.named_steps['model']

feature_names = preprocessor.get_feature_names_out()
coefficients = model.coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients,
    "abs_coefficient": np.abs(coefficients)
})

In [66]:
coef_df.shape
coef_df.sort_values("coefficient", ascending=False).head(30)

,feature,coefficient,abs_coefficient
11431,cat__card_addr_15063_nan,7.722047,7.722047
27775,cat__card_addr_6457_325.0,6.539311,6.539311
20978,cat__card_addr_2939_204.0,6.399613,6.399613
15489,cat__card_addr_16927_203.0,6.357897,6.357897
3247,cat__card_addr_11233_325.0,6.248562,6.248562
23628,cat__card_addr_4347_269.0,6.233026,6.233026
26416,cat__card_addr_5853_181.0,6.219047,6.219047
29284,cat__card_addr_7207_441.0,6.175477,6.175477
24242,cat__card_addr_4693_308.0,6.138248,6.138248
33897,cat__card_addr_9375_420.0,6.097948,6.097948


In [71]:
result = permutation_importance(
    pipe, X_val, y_val, n_repeats=10, random_state=42, n_jobs=-1
)

In [74]:
importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance_Mean': result.importances_mean,
    'Importance_Std': result.importances_std
})

importance_df = importance_df.sort_values(by='Importance_Mean', ascending=False)

In [78]:
importance_df.head(20)

,Feature,Importance_Mean,Importance_Std
2,ProductCD,0.014170,0.000243
36,card_addr,0.013769,0.000964
29,P_emaildomain_provider,0.013215,0.000293
31,R_emaildomain_provider,0.012997,0.000616
18,missing_id_count,0.012767,0.000351
14,missing_V_count,0.008367,0.000425
30,R_emaildomain_is_missing,0.007007,0.000425
35,email_product,0.006641,0.000546
8,card6,0.006373,0.000479
24,transaction_amt_log,0.006051,0.000482
